# Task 3, Step 6b: Actual Speculative Execution After Distillation

The same student-written exact sampler and the same EOS-aware 120-prompt protocol as the baseline (gamma in {2, 4},
two paired repetitions, 64-token cap, adjacent counterbalanced target-only timing) for the four merged drafts; the
original draft's rows come from notebook 2. Per round, synchronised timers split draft proposal, target verification,
acceptance / residual sampling and cache rollback; prefill and host overhead are reported separately.
Cost model: c and v_gamma from isolated step timings of each draft, alpha = empirical accepted/examined, predicted
S_gamma = (1 + sum_j alpha^j) / (gamma c + v_gamma), measured S_gamma = speculative / target-only tokens per second,
and the break-even alpha. One unmerged adapter is timed separately (wrapper overhead; not the primary comparison).

In [ ]:
# --- Setup: locate (or clone) the PA4 repo, pin library versions, prepare data -------------
import os, sys, subprocess, importlib.metadata as md
REPO_URL = "https://github.com/moizejaz-31/ATML-PA4.git"   # change if your repo lives elsewhere

def _find_root(d):
    d = os.path.abspath(d)
    while True:
        if os.path.isdir(os.path.join(d, "shared")) and os.path.isdir(os.path.join(d, "task3_speculative")):
            return d
        if os.path.dirname(d) == d:
            return None
        d = os.path.dirname(d)

ROOT = _find_root(os.getcwd())
if ROOT is None:                                   # fresh Kaggle / Colab runtime
    subprocess.run(["git", "clone", "-q", REPO_URL, "ATML-PA4"], check=True)
    ROOT = os.path.abspath("ATML-PA4")
os.chdir(ROOT)
sys.path.insert(0, ROOT)

def _ver(p):
    try:
        return md.version(p)
    except md.PackageNotFoundError:
        return None

if _ver("transformers") != "4.57.1" or _ver("huggingface_hub") != "0.36.2" or _ver("peft") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "transformers==4.57.1", "huggingface_hub==0.36.2",
                    "datasets>=3,<5", "peft>=0.13"], check=True)
    print(">>> Installed pinned versions. Restart the kernel, then run this cell again.")

from shared.src.nb import init_notebook
ctx = init_notebook("task3", need_public=True, need_prefix_banks=False)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from shared.src.io import read_jsonl, write_jsonl, save_json
from shared.src.plotting import save_figure
from task3_speculative.src.data import test_prompts
from task3_speculative.src.distill import load_models, load_merged, load_unmerged
from task3_speculative.src.spec_eval import run_benchmark, isolated_step_costs, predicted_speedup, break_even_alpha

M = load_models()
tok, target, original = M["tok"], M["target"], M["draft"]
MERGED = ctx.checkpoints / "merged"
OUT = ctx.results / "spec"
STUDENTS = [f"{m}_s{s}" for m in ("off", "on") for s in (6304, 6305)]
drafts = {n: load_merged(MERGED / n) for n in STUDENTS}

In [ ]:
bench_path = OUT / "benchmark_eos64.jsonl"
run_benchmark(target, drafts, tok, bench_path)
math24 = set([p["pid"] for p in test_prompts() if p["domain"] == "math"][:24])
run_benchmark(target, drafts, tok, OUT / "benchmark_fixed64_ignore_eos.jsonl", ignore_eos=True, prompt_filter=lambda p: p["pid"] in math24)

## Unmerged adapter (PEFT wrapper) timing for one draft, gamma = 4, first repetition

In [ ]:
unm_path = OUT / "benchmark_unmerged_on_s6304.jsonl"
unm = load_unmerged(ctx.checkpoints / "distill" / "on_policy_s6304" / "adapter")
run_benchmark(target, {"on_s6304_unmerged": unm}, tok, unm_path, gammas=(4,), reps=(0,))
del unm; torch.cuda.empty_cache()

## Isolated step costs per draft and the cost model

In [ ]:
iso_path = OUT / "isolated_step_costs_all.jsonl"
if not iso_path.exists():
    rows = []
    for n, d in {"original": original, **drafts}.items():
        for r in isolated_step_costs(target, d, tok):
            rows.append(dict(draft=n, **r))
    write_jsonl(iso_path, rows)
iso = pd.DataFrame(read_jsonl(iso_path))
iso.groupby("draft")[["target_step_ms", "draft_step_ms", "c", "v_2", "v_4"]].mean().round(3)

In [ ]:
b = pd.DataFrame(read_jsonl(bench_path))
cm = iso.groupby("draft")[["c", "v_2", "v_4"]].mean()
rows = []
for (d, g), x in b.groupby(["draft", "gamma"]):
    alpha = x.accepted.sum() / x.examined.sum()
    c, v = cm.loc[d, "c"], cm.loc[d, f"v_{g}"]
    meas = x.tokens_per_s.mean() / x.base_tokens_per_s.mean()
    meas_dec = (x.n_out.sum() / x.decode_ms.sum()) / (x.base_n_out.sum() / x.base_decode_ms.sum())
    rows.append(dict(draft=d, gamma=g, alpha=alpha, c=c, v=v, predicted_S=predicted_speedup(alpha, g, c, v),
                     measured_S_total=meas, measured_S_decode_only=meas_dec, break_even_alpha=break_even_alpha(g, c, v),
                     acc_per_proposed=x.accepted.sum() / x.proposed.sum(), target_passes_per_token=x.target_passes.sum() / x.n_out.sum(),
                     draft_passes_per_token=x.draft_forward_passes.sum() / x.n_out.sum(), mean_out=x.n_out.mean(),
                     eos_frac=x.eos.mean(), truncated_frac=x.truncated.mean(), peak_alloc_mb=x.peak_allocated_mb.mean(),
                     base_peak_alloc_mb=x.base_peak_allocated_mb.mean()))
cost = pd.DataFrame(rows)
cost.to_csv(OUT / "cost_model_vs_measured.csv", index=False)
cost.round(3)

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(14, 4.5))
for g, m in ((2, "o"), (4, "s")):
    s = cost[cost.gamma == g]
    ax[0].scatter(s.predicted_S, s.measured_S_total, marker=m, label=f"gamma={g}")
    for _, r in s.iterrows():
        ax[0].annotate(r.draft, (r.predicted_S, r.measured_S_total), fontsize=7)
lim = [0, max(cost.predicted_S.max(), cost.measured_S_total.max()) * 1.1]
ax[0].plot(lim, lim, "k--", lw=1); ax[0].axhline(1, c="gray", lw=0.8)
ax[0].set_xlabel("predicted speedup (cost model)"); ax[0].set_ylabel("measured speedup (tokens/s ratio)"); ax[0].legend()
parts = ["prefill_ms", "draft_ms", "verify_ms", "accept_ms", "rollback_ms", "host_overhead_ms"]
dec = b.groupby(["draft", "gamma"])[parts].mean()
(dec.div(b.groupby(["draft", "gamma"]).n_out.mean(), axis=0)).plot(kind="bar", stacked=True, ax=ax[1])
ax[1].set_ylabel("ms per output token"); ax[1].set_title("Per-round time decomposition (EOS-aware, 64 cap)")
save_figure(ctx.figures, "speculative_cost_model_and_decomposition")
dec.round(1)

In [ ]:
u = pd.DataFrame(read_jsonl(unm_path))
mg = b[(b.draft == "on_s6304") & (b.gamma == 4) & (b.rep == 0)]
pd.DataFrame({"merged": mg[["draft_ms", "total_ms", "tokens_per_s", "acc_per_examined"]].mean(),
              "unmerged (PEFT)": u[["draft_ms", "total_ms", "tokens_per_s", "acc_per_examined"]].mean()}).round(2)